In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_period AS
WITH extract_month AS (
    SELECT DISTINCT date_trunc('month', date) AS base_month
    FROM bakery_pos_develop.silver.account_move_line
    WHERE date IS NOT NULL
)
SELECT 
    CAST(date_format(base_month, 'yyyyMM') AS INT) AS period_sk,
    month(base_month) AS month,
    year(base_month) AS year,
    CAST(base_month AS DATE) AS start_date,
    CAST(last_day(base_month) AS DATE) AS end_date
FROM extract_month;


In [0]:
%sql

SELECT * FROM stg_period

In [0]:
%sql
drop table if exists bakery_pos_develop.gold.dim_period

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_period (
    period_sk INT,
    month INT,
    year INT,
    start_date DATE,
    end_date DATE,
    PRIMARY KEY (period_sk)
) USING DELTA;


In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_period AS target
USING (
    SELECT * FROM stg_period
) AS source
ON target.period_sk = source.period_sk
WHEN NOT MATCHED THEN
    INSERT (
        period_sk, 
        month, 
        year, 
        start_date, 
        end_date
    )
    VALUES (
        source.period_sk, 
        source.month, 
        source.year, 
        source.start_date, 
        source.end_date
    );